# CIRA-CIC-DoHBrw-2020 Layer 1: reviewed research notebook

Derived from the notebook supplied by the author on 9 November 2026. Saved outputs and temporary dataset-upload cells have been removed. See `docs/REVIEW.md` for changes, limitations and original result evidence. This cleaned copy has not been benchmark-rerun.


This notebook bypasses local laptop storage and processing.

It will:

1. install the required Python packages in the cloud runtime;
2. retrieve only `L1-DoH-NonDoH.parquet` from the public Kaggle dataset;
3. verify the agreed record counts and data quality;
4. run Logistic Regression, a 100-tree Random Forest and a 200-round XGBoost model;
5. apply an 80:20 split, three-fold validation and three fixed seeds;
6. create a compact ZIP archive containing the measured results.

Use **Runtime > Run all** in Google Colab, or import this notebook into Kaggle and run all cells.

The parquet file remains inside the temporary cloud runtime. It is not downloaded to your laptop.

## Three-seed, three-fold protocol

This version retains all 1,019,117 records but reduces repeated computation.

| Component | Faster setting |
|---|---:|
| Random Forest | 100 trees |
| XGBoost | 200 boosting rounds, depth 5 |
| Cross-validation | Stratified 3-fold |
| Repeated seeds | 42, 52 and 62 |
| Permutation sample | 20,000 held-out records |
| Permutation repeats | 3 |

This reduces the experiment from about 90 model fits to about 36 model fits. The settings must be reported exactly in the paper.

## 1. Install the required packages

The installation occurs only inside the cloud virtual machine.

In [ ]:
%pip install -q kagglehub pyarrow scipy scikit-learn xgboost matplotlib joblib requests

## 2. Obtain the cleaned Layer 1 parquet file

This uses the `dhoogla/cicdohbrw2020` Kaggle derivative. Its version is not pinned in the original notebook. Record the file SHA-256 and compare it with the original experiment audit before claiming the same input. Authenticate through KaggleHub only when needed; never save a token in this notebook.


In [ ]:
from pathlib import Path
import kagglehub

# Uncomment only if the public download requests authentication.
# kagglehub.login()

downloaded = kagglehub.dataset_download(
    "dhoogla/cicdohbrw2020",
    path="L1-DoH-NonDoH.parquet",
)

candidate = Path(downloaded)
if candidate.is_file():
    DATA_PATH = candidate
else:
    matches = list(candidate.rglob("L1-DoH-NonDoH.parquet"))
    if not matches:
        # Fallback search in standard Kaggle and Colab locations.
        search_roots = [Path("/kaggle/input"), Path("/content"), Path.home() / ".cache" / "kagglehub"]
        for root in search_roots:
            if root.exists():
                matches.extend(root.rglob("L1-DoH-NonDoH.parquet"))
    if not matches:
        raise FileNotFoundError("L1-DoH-NonDoH.parquet was not found after the KaggleHub download.")
    if len(matches) != 1:
        raise RuntimeError("Multiple parquet candidates found; set DATA_PATH explicitly.")
    DATA_PATH = matches[0]

print("Dataset path:", DATA_PATH)
print("File size:", f"{DATA_PATH.stat().st_size / (1024**2):,.2f} MiB")

## 3. Verify the dataset before modelling

The notebook stops if the file does not match the previously measured Layer 1 dataset.

In [ ]:
import pandas as pd
import numpy as np

df_check = pd.read_parquet(DATA_PATH)

expected_rows = 1_019_117
expected_columns = 30
expected_counts = {"NonDoH": 750_456, "DoH": 268_661}

actual_counts = df_check["Label"].value_counts().to_dict()
missing_cells = int(df_check.isna().sum().sum())
duplicate_rows = int(df_check.duplicated().sum())
predictors = [c for c in df_check.columns if c != "Label"]
infinite_cells = int(np.isinf(df_check[predictors].to_numpy(dtype=np.float64, copy=False)).sum())

checks = {
    "rows": (len(df_check), expected_rows),
    "columns": (df_check.shape[1], expected_columns),
    "predictors": (len(predictors), 29),
    "label_counts": (actual_counts, expected_counts),
    "missing_cells": (missing_cells, 0),
    "duplicate_rows": (duplicate_rows, 0),
    "infinite_cells": (infinite_cells, 0),
}

for name, (actual, expected) in checks.items():
    status = "PASS" if actual == expected else "FAIL"
    print(f"{name:18s} {status:4s} | actual={actual} | expected={expected}")

failed = [name for name, (actual, expected) in checks.items() if actual != expected]
if failed:
    raise ValueError(f"Dataset validation failed: {failed}")

print("\nDataset validation passed. The full experiment may proceed.")
del df_check

## 4. Create the locked experiment runner

This cell writes the agreed reproducible experiment programme into the cloud workspace.

In [ ]:
from pathlib import Path

RUNNER_PATH = Path('/content/run_experiment.py') if Path('/content').exists() else Path('run_experiment.py')
RUNNER_PATH.write_text('#!/usr/bin/env python3\n"""\nReviewed derivative of the supplied CIRA-CIC-DoHBrw-2020 Layer 1 runner.\nOriginal model settings and numerical evaluation are retained. See docs/REVIEW.md.\n\nTask:\n    DoH versus NonDoH HTTPS traffic classification.\n\nModels:\n    Logistic Regression, Random Forest and XGBoost.\n\nProtocol:\n    Stratified 80% development and 20% untouched test split.\n    Stratified three-fold cross-validation within the development set.\n    Three fixed seeds: 42, 52 and 62.\n    Class weighting is derived from the development data.\n    Standardisation is fitted within the Logistic Regression pipeline only.\n\nThe script never fabricates or fills experimental results. It validates the\ndataset before model fitting and writes all measured outputs to disk.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport platform\nimport sys\nimport time\nimport warnings\nimport zipfile\nfrom pathlib import Path\nfrom typing import Any\n\nimport joblib\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport requests\nimport scipy\nimport sklearn\nimport xgboost\nfrom scipy import stats\nfrom sklearn.base import clone\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.inspection import permutation_importance\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    ConfusionMatrixDisplay,\n    accuracy_score,\n    average_precision_score,\n    balanced_accuracy_score,\n    confusion_matrix,\n    f1_score,\n    matthews_corrcoef,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n)\nfrom sklearn.model_selection import (\n    StratifiedKFold,\n    cross_validate,\n    train_test_split,\n)\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.utils.class_weight import compute_class_weight\nfrom xgboost import XGBClassifier\n\n\nEXPECTED_ROWS = 1_019_117\nEXPECTED_COLUMNS = 30\nEXPECTED_FEATURES = 29\nEXPECTED_LABEL_COUNTS = {"DoH": 268_661, "NonDoH": 750_456}\nSEEDS = [42, 52, 62]\nPOSITIVE_LABEL = "DoH"\nNEGATIVE_LABEL = "NonDoH"\nTARGET_COLUMN = "Label"\nPARQUET_NAME = "L1-DoH-NonDoH.parquet"\n\nDOWNLOAD_URLS = [\n    (\n        "ResearchGate archive",\n        "https://www.researchgate.net/profile/Laurens-Dhooge/"\n        "publication/367633971_CIRA-CIC-DoHBrw2020-Improved/"\n        "links/63d954be62d2a24f92e27561/"\n        "CIRA-CIC-DoHBrw2020-Improved.zip",\n    ),\n    (\n        "Kaggle public dataset API",\n        "https://www.kaggle.com/api/v1/datasets/download/"\n        "dhoogla/cicdohbrw2020",\n    ),\n]\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description="Run the CIRA-CIC-DoHBrw-2020 Layer 1 experiment."\n    )\n    parser.add_argument(\n        "--data",\n        type=Path,\n        default=None,\n        help="Path to L1-DoH-NonDoH.parquet.",\n    )\n    parser.add_argument(\n        "--workdir",\n        type=Path,\n        default=Path("cira_doh_run"),\n        help="Directory for downloaded data and experimental outputs.",\n    )\n    parser.add_argument(\n        "--download",\n        action="store_true",\n        help="Download the public improved archive when the parquet file is absent.",\n    )\n    parser.add_argument(\n        "--n-jobs",\n        type=int,\n        default=-1,\n        help="Threads used within Random Forest and XGBoost. Default: all available.",\n    )\n    parser.add_argument(\n        "--seeds",\n        type=int,\n        nargs="+",\n        default=SEEDS,\n        help="Repeated split and model seeds.",\n    )\n    parser.add_argument(\n        "--skip-cv",\n        action="store_true",\n        help="Skip three-fold development cross-validation. Not suitable for the paper run.",\n    )\n    parser.add_argument(\n        "--skip-importance",\n        action="store_true",\n        help="Skip permutation importance.",\n    )\n    parser.add_argument(\n        "--importance-sample",\n        type=int,\n        default=20_000,\n        help="Maximum held-out records used for permutation importance.",\n    )\n    parser.add_argument(\n        "--importance-repeats",\n        type=int,\n        default=3,\n        help="Permutation repetitions for each feature.",\n    )\n    parser.add_argument(\n        "--save-models",\n        action="store_true",\n        help="Save fitted seed-42 estimators with joblib.",\n    )\n    parser.add_argument(\n        "--smoke-test",\n        action="store_true",\n        help=(\n            "Use a stratified sample of 50,000 records and the first seed only. "\n            "Outputs are diagnostic and must not be reported as paper results."\n        ),\n    )\n    return parser.parse_args()\n\n\ndef json_default(value: Any) -> Any:\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (np.floating,)):\n        return float(value)\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, np.ndarray):\n        return value.tolist()\n    raise TypeError(f"Cannot serialise {type(value)!r}")\n\n\ndef write_json(path: Path, payload: dict[str, Any]) -> None:\n    path.write_text(\n        json.dumps(payload, indent=2, default=json_default),\n        encoding="utf-8",\n    )\n\n\ndef sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        while chunk := handle.read(chunk_size):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef safe_extract_zip(archive: Path, destination: Path) -> None:\n    destination = destination.resolve()\n    with zipfile.ZipFile(archive) as zf:\n        for member in zf.infolist():\n            target = (destination / member.filename).resolve()\n            if destination not in target.parents and target != destination:\n                raise RuntimeError(f"Unsafe archive member: {member.filename}")\n        zf.extractall(destination)\n\n\ndef stream_download(url: str, destination: Path) -> None:\n    headers = {\n        "User-Agent": (\n            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "\n            "AppleWebKit/537.36 Chrome/150 Safari/537.36"\n        )\n    }\n    with requests.get(\n        url,\n        headers=headers,\n        stream=True,\n        timeout=(30, 300),\n        allow_redirects=True,\n    ) as response:\n        response.raise_for_status()\n        content_type = response.headers.get("content-type", "").lower()\n        with destination.open("wb") as handle:\n            for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):\n                if chunk:\n                    handle.write(chunk)\n\n    if destination.stat().st_size < 1_000_000:\n        head = destination.read_bytes()[:300].lower()\n        if b"<html" in head or b"<!doctype html" in head:\n            destination.unlink(missing_ok=True)\n            raise RuntimeError(\n                f"The server returned an HTML page rather than the archive "\n                f"(content type: {content_type or \'unknown\'})."\n            )\n\n\ndef locate_or_download_dataset(\n    requested_path: Path | None,\n    workdir: Path,\n    permit_download: bool,\n) -> Path:\n    candidates: list[Path] = []\n    if requested_path is not None:\n        if not requested_path.is_file():\n            raise FileNotFoundError(f"Explicit dataset path does not exist: {requested_path}")\n        return requested_path.resolve()\n    candidates.extend(\n        [\n            Path(PARQUET_NAME),\n            workdir / PARQUET_NAME,\n            workdir / "data" / PARQUET_NAME,\n        ]\n    )\n\n    for candidate in candidates:\n        if candidate.is_file():\n            return candidate.resolve()\n\n    recursively_found = list(workdir.rglob(PARQUET_NAME))\n    if recursively_found:\n        return recursively_found[0].resolve()\n\n    if not permit_download:\n        raise FileNotFoundError(\n            f"{PARQUET_NAME} was not found. Supply --data PATH or add --download."\n        )\n\n    data_dir = workdir / "data"\n    data_dir.mkdir(parents=True, exist_ok=True)\n    errors: list[str] = []\n\n    for source_name, url in DOWNLOAD_URLS:\n        archive = data_dir / "CIC-DoHBrw2020-Improved.zip"\n        archive.unlink(missing_ok=True)\n        try:\n            print(f"Downloading from {source_name}...")\n            stream_download(url, archive)\n            if not zipfile.is_zipfile(archive):\n                raise RuntimeError("Downloaded object is not a valid ZIP archive.")\n            safe_extract_zip(archive, data_dir)\n            found = list(data_dir.rglob(PARQUET_NAME))\n            if not found:\n                raise FileNotFoundError(\n                    f"{PARQUET_NAME} was not present in the downloaded archive."\n                )\n            return found[0].resolve()\n        except Exception as exc:\n            errors.append(f"{source_name}: {exc}")\n            archive.unlink(missing_ok=True)\n\n    raise RuntimeError(\n        "The public archive could not be retrieved automatically. "\n        "Place L1-DoH-NonDoH.parquet in the working directory.\\n"\n        + "\\n".join(errors)\n    )\n\n\ndef audit_dataset(df: pd.DataFrame, dataset_path: Path) -> tuple[list[str], pd.Series]:\n    label_counts = df[TARGET_COLUMN].value_counts(dropna=False).to_dict()\n    feature_columns = [c for c in df.columns if c != TARGET_COLUMN]\n\n    numeric_frame = df[feature_columns]\n    non_numeric = [\n        c for c in feature_columns if not pd.api.types.is_numeric_dtype(df[c])\n    ]\n    missing_cells = int(df.isna().sum().sum())\n    duplicate_rows = int(df.duplicated().sum())\n    infinite_cells = int(\n        np.isinf(numeric_frame.select_dtypes(include=[np.number]).to_numpy(dtype=np.float64, copy=False)).sum()\n    )\n    constant_features = [\n        c for c in feature_columns if df[c].nunique(dropna=False) <= 1\n    ]\n    near_constant_features = []\n    for column in feature_columns:\n        proportions = df[column].value_counts(normalize=True, dropna=False)\n        if not proportions.empty and float(proportions.iloc[0]) >= 0.999:\n            near_constant_features.append(\n                {\n                    "feature": column,\n                    "dominant_proportion": float(proportions.iloc[0]),\n                }\n            )\n\n    audit = {\n        "dataset_file": dataset_path.name,\n        "sha256": sha256_file(dataset_path),\n        "file_size_bytes": dataset_path.stat().st_size,\n        "rows": int(df.shape[0]),\n        "columns": int(df.shape[1]),\n        "predictor_count": len(feature_columns),\n        "target": TARGET_COLUMN,\n        "label_counts": label_counts,\n        "missing_cells": missing_cells,\n        "exact_duplicate_rows": duplicate_rows,\n        "infinite_numeric_cells": infinite_cells,\n        "non_numeric_predictors": non_numeric,\n        "constant_features": constant_features,\n        "near_constant_features_99_9_percent": near_constant_features,\n        "dtypes": {c: str(dtype) for c, dtype in df.dtypes.items()},\n    }\n\n    failures = []\n    if df.shape[0] != EXPECTED_ROWS:\n        failures.append(f"rows={df.shape[0]}, expected {EXPECTED_ROWS}")\n    if df.shape[1] != EXPECTED_COLUMNS:\n        failures.append(f"columns={df.shape[1]}, expected {EXPECTED_COLUMNS}")\n    if len(feature_columns) != EXPECTED_FEATURES:\n        failures.append(\n            f"predictors={len(feature_columns)}, expected {EXPECTED_FEATURES}"\n        )\n    if label_counts != EXPECTED_LABEL_COUNTS:\n        failures.append(\n            f"label_counts={label_counts}, expected {EXPECTED_LABEL_COUNTS}"\n        )\n    if missing_cells != 0:\n        failures.append(f"missing_cells={missing_cells}, expected 0")\n    if duplicate_rows != 0:\n        failures.append(f"duplicate_rows={duplicate_rows}, expected 0")\n    if infinite_cells != 0:\n        failures.append(f"infinite_cells={infinite_cells}, expected 0")\n    if non_numeric:\n        failures.append(f"non_numeric_predictors={non_numeric}")\n    if constant_features:\n        failures.append(f"constant_features={constant_features}")\n    if near_constant_features:\n        failures.append(f"near_constant_features={near_constant_features}")\n\n    if failures:\n        audit["validation_status"] = "FAILED"\n        audit["validation_failures"] = failures\n    else:\n        audit["validation_status"] = "PASSED"\n        audit["validation_failures"] = []\n\n    return feature_columns, pd.Series(audit, dtype=object)\n\n\ndef build_models(seed: int, scale_pos_weight: float, n_jobs: int) -> dict[str, Any]:\n    logistic = Pipeline(\n        steps=[\n            ("standardise", StandardScaler()),\n            (\n                "classifier",\n                LogisticRegression(\n                    C=1.0,\n                    solver="lbfgs",\n                    max_iter=1_000,\n                    class_weight="balanced",\n                    random_state=seed,\n                ),\n            ),\n        ]\n    )\n\n    random_forest = RandomForestClassifier(\n        n_estimators=100,\n        criterion="gini",\n        max_depth=None,\n        min_samples_split=2,\n        min_samples_leaf=1,\n        max_features="sqrt",\n        bootstrap=True,\n        class_weight="balanced_subsample",\n        n_jobs=n_jobs,\n        random_state=seed,\n    )\n\n    xgboost_model = XGBClassifier(\n        n_estimators=200,\n        learning_rate=0.05,\n        max_depth=5,\n        min_child_weight=1,\n        subsample=0.8,\n        colsample_bytree=0.8,\n        gamma=0.0,\n        reg_alpha=0.0,\n        reg_lambda=1.0,\n        objective="binary:logistic",\n        eval_metric="logloss",\n        tree_method="hist",\n        scale_pos_weight=scale_pos_weight,\n        n_jobs=n_jobs,\n        random_state=seed,\n        verbosity=1,\n    )\n\n    return {\n        "Logistic Regression": logistic,\n        "Random Forest": random_forest,\n        "XGBoost": xgboost_model,\n    }\n\n\ndef probability_scores(estimator: Any, X: pd.DataFrame) -> np.ndarray:\n    if hasattr(estimator, "predict_proba"):\n        return np.asarray(estimator.predict_proba(X))[:, 1]\n    if hasattr(estimator, "decision_function"):\n        scores = np.asarray(estimator.decision_function(X))\n        return 1.0 / (1.0 + np.exp(-scores))\n    raise TypeError("Estimator provides neither predict_proba nor decision_function.")\n\n\ndef calculate_test_metrics(\n    y_true: pd.Series,\n    predictions: np.ndarray,\n    probabilities: np.ndarray,\n) -> tuple[dict[str, float], np.ndarray]:\n    cm = confusion_matrix(y_true, predictions, labels=[0, 1])\n    tn, fp, fn, tp = cm.ravel()\n    fpr = fp / (fp + tn) if (fp + tn) else np.nan\n\n    metrics = {\n        "accuracy": accuracy_score(y_true, predictions),\n        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),\n        "precision_doh": precision_score(\n            y_true, predictions, pos_label=1, zero_division=0\n        ),\n        "recall_doh": recall_score(\n            y_true, predictions, pos_label=1, zero_division=0\n        ),\n        "macro_f1": f1_score(y_true, predictions, average="macro"),\n        "mcc": matthews_corrcoef(y_true, predictions),\n        "pr_auc": average_precision_score(y_true, probabilities),\n        "roc_auc": roc_auc_score(y_true, probabilities),\n        "false_positive_rate": fpr,\n        "tn": int(tn),\n        "fp": int(fp),\n        "fn": int(fn),\n        "tp": int(tp),\n    }\n    return metrics, cm\n\n\ndef summarise_repetitions(results: pd.DataFrame) -> pd.DataFrame:\n    metric_columns = [\n        "accuracy",\n        "balanced_accuracy",\n        "precision_doh",\n        "recall_doh",\n        "macro_f1",\n        "mcc",\n        "pr_auc",\n        "roc_auc",\n        "false_positive_rate",\n        "training_time_seconds",\n        "prediction_time_seconds",\n    ]\n    rows = []\n    for model_name, group in results.groupby("model", sort=False):\n        n = len(group)\n        t_critical = stats.t.ppf(0.975, df=n - 1) if n > 1 else np.nan\n        for metric in metric_columns:\n            values = group[metric].astype(float)\n            mean = float(values.mean())\n            std = float(values.std(ddof=1)) if n > 1 else np.nan\n            ci_half = (\n                float(t_critical * std / np.sqrt(n))\n                if n > 1 and np.isfinite(std)\n                else np.nan\n            )\n            rows.append(\n                {\n                    "model": model_name,\n                    "metric": metric,\n                    "n_repetitions": n,\n                    "mean": mean,\n                    "standard_deviation": std,\n                    "ci95_lower": mean - ci_half if np.isfinite(ci_half) else np.nan,\n                    "ci95_upper": mean + ci_half if np.isfinite(ci_half) else np.nan,\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef create_plots(\n    test_results: pd.DataFrame,\n    summary: pd.DataFrame,\n    confusion_totals: dict[str, np.ndarray],\n    importance_frame: pd.DataFrame | None,\n    output_dir: Path,\n) -> None:\n    plot_dir = output_dir / "plots"\n    plot_dir.mkdir(parents=True, exist_ok=True)\n\n    for metric in ["macro_f1", "mcc", "pr_auc", "false_positive_rate"]:\n        metric_summary = summary[summary["metric"] == metric].copy()\n        fig, ax = plt.subplots(figsize=(7.5, 4.8))\n        errors = np.vstack(\n            [\n                metric_summary["mean"] - metric_summary["ci95_lower"],\n                metric_summary["ci95_upper"] - metric_summary["mean"],\n            ]\n        )\n        ax.bar(\n            metric_summary["model"],\n            metric_summary["mean"],\n            yerr=errors,\n            capsize=5,\n        )\n        ax.set_ylabel(metric.replace("_", " ").upper())\n        ax.set_title(f"Test {metric.replace(\'_\', \' \').title()} across three seeds")\n        ax.tick_params(axis="x", rotation=15)\n        fig.tight_layout()\n        fig.savefig(plot_dir / f"test_{metric}.png", dpi=220)\n        plt.close(fig)\n\n    for model_name, matrix in confusion_totals.items():\n        fig, ax = plt.subplots(figsize=(5.8, 5.0))\n        display = ConfusionMatrixDisplay(\n            confusion_matrix=matrix,\n            display_labels=[NEGATIVE_LABEL, POSITIVE_LABEL],\n        )\n        display.plot(ax=ax, values_format="d")\n        ax.set_title(f"{model_name}, summed test confusion matrix")\n        fig.tight_layout()\n        safe_name = model_name.lower().replace(" ", "_")\n        fig.savefig(plot_dir / f"confusion_{safe_name}.png", dpi=220)\n        plt.close(fig)\n\n    if importance_frame is not None and not importance_frame.empty:\n        for model_name, group in importance_frame.groupby("model", sort=False):\n            top = group.sort_values("importance_mean", ascending=False).head(15)\n            top = top.sort_values("importance_mean", ascending=True)\n            fig, ax = plt.subplots(figsize=(8.5, 6.5))\n            ax.barh(\n                top["feature"],\n                top["importance_mean"],\n                xerr=top["importance_std"],\n                capsize=3,\n            )\n            ax.set_xlabel("Decrease in held-out Average Precision after permutation")\n            ax.set_title(f"{model_name}, permutation importance")\n            fig.tight_layout()\n            safe_name = model_name.lower().replace(" ", "_")\n            fig.savefig(\n                plot_dir / f"permutation_importance_{safe_name}.png",\n                dpi=220,\n            )\n            plt.close(fig)\n\n\ndef main() -> int:\n    args = parse_args()\n    if args.importance_sample < 1 or args.importance_repeats < 1:\n        raise ValueError("Importance sample and repeats must be positive.")\n    if args.n_jobs == 0 or len(args.seeds) != len(set(args.seeds)):\n        raise ValueError("n-jobs cannot be zero and seeds must be unique.")\n    args.workdir.mkdir(parents=True, exist_ok=True)\n    output_dir = args.workdir / "outputs"\n    if output_dir.exists() and any(output_dir.iterdir()):\n        raise FileExistsError("Output directory is not empty. Use a new --workdir to avoid mixing runs.")\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    warnings.filterwarnings("default")\n    start_wall = time.time()\n\n    dataset_path = locate_or_download_dataset(\n        args.data,\n        args.workdir,\n        args.download,\n    )\n    print(f"Reading {dataset_path}...")\n    df = pd.read_parquet(dataset_path)\n\n    feature_columns, audit_series = audit_dataset(df, dataset_path)\n    audit = audit_series.to_dict()\n    write_json(output_dir / "dataset_audit.json", audit)\n    if audit["validation_status"] != "PASSED":\n        raise ValueError(\n            "Dataset validation failed. See outputs/dataset_audit.json. "\n            + "; ".join(audit["validation_failures"])\n        )\n\n    feature_summary = df[feature_columns].describe(\n        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]\n    ).T\n    feature_summary["dtype"] = [str(df[c].dtype) for c in feature_summary.index]\n    feature_summary["unique_values"] = [\n        int(df[c].nunique(dropna=False)) for c in feature_summary.index\n    ]\n    feature_summary.to_csv(output_dir / "feature_summary.csv", index_label="feature")\n\n    if args.smoke_test:\n        print("SMOKE TEST: results are not suitable for the paper.")\n        df, _ = train_test_split(\n            df,\n            train_size=min(50_000, len(df)),\n            stratify=df[TARGET_COLUMN],\n            random_state=args.seeds[0],\n        )\n        active_seeds = [args.seeds[0]]\n    else:\n        active_seeds = args.seeds\n\n    X = df[feature_columns]\n    y = df[TARGET_COLUMN].map({NEGATIVE_LABEL: 0, POSITIVE_LABEL: 1}).astype(np.int8)\n\n    cv_rows: list[dict[str, Any]] = []\n    test_rows: list[dict[str, Any]] = []\n    confusion_totals: dict[str, np.ndarray] = {}\n    seed_42_models: dict[str, Any] = {}\n    seed_42_test: tuple[pd.DataFrame, pd.Series] | None = None\n    config_record: dict[str, Any] = {}\n\n    scoring = {\n        "accuracy": "accuracy",\n        "balanced_accuracy": "balanced_accuracy",\n        "precision_doh": "precision",\n        "recall_doh": "recall",\n        "macro_f1": "f1_macro",\n        "mcc": "matthews_corrcoef",\n        "pr_auc": "average_precision",\n        "roc_auc": "roc_auc",\n    }\n\n    for seed in active_seeds:\n        print(f"\\nSeed {seed}: creating stratified 80/20 split...")\n        X_dev, X_test, y_dev, y_test = train_test_split(\n            X,\n            y,\n            test_size=0.20,\n            stratify=y,\n            random_state=seed,\n        )\n\n        class_weights = compute_class_weight(\n            class_weight="balanced",\n            classes=np.array([0, 1]),\n            y=y_dev,\n        )\n        negative_count = int((y_dev == 0).sum())\n        positive_count = int((y_dev == 1).sum())\n        scale_pos_weight = negative_count / positive_count\n\n        models = build_models(seed, scale_pos_weight, args.n_jobs)\n        if not config_record:\n            config_record = {\n                "seeds": active_seeds,\n                "split": {\n                    "development_fraction": 0.80,\n                    "test_fraction": 0.20,\n                    "stratified": True,\n                    "seed_controls_split_and_estimator": True,\n                },\n                "cross_validation": {\n                    "folds": 3,\n                    "stratified": True,\n                    "shuffle": True,\n                    "performed": not args.skip_cv,\n                },\n                "class_imbalance": {\n                    "negative_class": NEGATIVE_LABEL,\n                    "positive_class": POSITIVE_LABEL,\n                    "logistic_regression": "class_weight=\'balanced\'",\n                    "random_forest": "class_weight=\'balanced_subsample\'",\n                    "xgboost": "scale_pos_weight = development NonDoH / DoH",\n                    "example_development_class_weights": {\n                        NEGATIVE_LABEL: float(class_weights[0]),\n                        POSITIVE_LABEL: float(class_weights[1]),\n                    },\n                    "example_xgboost_scale_pos_weight": scale_pos_weight,\n                },\n                "models": {\n                    name: estimator.get_params(deep=True)\n                    for name, estimator in models.items()\n                },\n                "decision_threshold": 0.5,\n                "positive_class": POSITIVE_LABEL,\n                "primary_metrics": ["macro_f1", "mcc", "pr_auc"],\n                "additional_metrics": [\n                    "accuracy",\n                    "balanced_accuracy",\n                    "precision_doh",\n                    "recall_doh",\n                    "roc_auc",\n                    "false_positive_rate",\n                    "training_time_seconds",\n                    "prediction_time_seconds",\n                ],\n            }\n\n        for model_name, estimator in models.items():\n            print(f"Seed {seed}, {model_name}")\n\n            if not args.skip_cv:\n                cv = StratifiedKFold(\n                    n_splits=3,\n                    shuffle=True,\n                    random_state=seed,\n                )\n                cv_result = cross_validate(\n                    clone(estimator),\n                    X_dev,\n                    y_dev,\n                    cv=cv,\n                    scoring=scoring,\n                    return_train_score=False,\n                    n_jobs=1,\n                    error_score="raise",\n                )\n                for fold_index in range(3):\n                    row = {\n                        "seed": seed,\n                        "model": model_name,\n                        "fold": fold_index + 1,\n                        "fit_time_seconds": cv_result["fit_time"][fold_index],\n                        "score_time_seconds": cv_result["score_time"][fold_index],\n                    }\n                    for metric_name in scoring:\n                        row[metric_name] = cv_result[\n                            f"test_{metric_name}"\n                        ][fold_index]\n                    cv_rows.append(row)\n\n            fit_start = time.perf_counter()\n            estimator.fit(X_dev, y_dev)\n            training_time = time.perf_counter() - fit_start\n\n            prediction_start = time.perf_counter()\n            predictions = estimator.predict(X_test)\n            probabilities = probability_scores(estimator, X_test)\n            prediction_time = time.perf_counter() - prediction_start\n\n            metric_values, cm = calculate_test_metrics(\n                y_test,\n                predictions,\n                probabilities,\n            )\n            test_rows.append(\n                {\n                    "seed": seed,\n                    "model": model_name,\n                    "development_records": len(X_dev),\n                    "test_records": len(X_test),\n                    "development_non_doh": negative_count,\n                    "development_doh": positive_count,\n                    "xgboost_scale_pos_weight": (\n                        scale_pos_weight if model_name == "XGBoost" else np.nan\n                    ),\n                    "training_time_seconds": training_time,\n                    "prediction_time_seconds": prediction_time,\n                    **metric_values,\n                }\n            )\n            confusion_totals.setdefault(model_name, np.zeros((2, 2), dtype=np.int64))\n            confusion_totals[model_name] += cm\n\n            if seed == 42:\n                seed_42_models[model_name] = estimator\n                seed_42_test = (X_test, y_test)\n\n    cv_frame = pd.DataFrame(cv_rows)\n    if not cv_frame.empty:\n        cv_frame.to_csv(output_dir / "cv_fold_metrics.csv", index=False)\n        cv_summary = (\n            cv_frame.groupby(["model"], sort=False)\n            .agg(\n                {\n                    col: ["mean", "std"]\n                    for col in [\n                        "accuracy",\n                        "balanced_accuracy",\n                        "precision_doh",\n                        "recall_doh",\n                        "macro_f1",\n                        "mcc",\n                        "pr_auc",\n                        "roc_auc",\n                        "fit_time_seconds",\n                        "score_time_seconds",\n                    ]\n                }\n            )\n        )\n        cv_summary.columns = [\n            "_".join(parts) for parts in cv_summary.columns.to_flat_index()\n        ]\n        cv_summary.reset_index().to_csv(\n            output_dir / "cv_metrics_summary.csv",\n            index=False,\n        )\n\n    test_frame = pd.DataFrame(test_rows)\n    test_frame.to_csv(output_dir / "test_metrics_by_seed.csv", index=False)\n    test_summary = summarise_repetitions(test_frame)\n    test_summary.to_csv(output_dir / "test_metrics_summary.csv", index=False)\n\n    confusion_rows = []\n    for model_name, matrix in confusion_totals.items():\n        tn, fp, fn, tp = matrix.ravel()\n        confusion_rows.append(\n            {\n                "model": model_name,\n                "summed_tn": int(tn),\n                "summed_fp": int(fp),\n                "summed_fn": int(fn),\n                "summed_tp": int(tp),\n            }\n        )\n    pd.DataFrame(confusion_rows).to_csv(\n        output_dir / "summed_confusion_matrices.csv",\n        index=False,\n    )\n\n    importance_frame: pd.DataFrame | None = None\n    if (\n        not args.skip_importance\n        and seed_42_models\n        and seed_42_test is not None\n    ):\n        X_test_42, y_test_42 = seed_42_test\n        sample_n = min(args.importance_sample, len(X_test_42))\n        if sample_n == len(X_test_42):\n            X_imp, y_imp = X_test_42, y_test_42\n        else:\n            X_imp, _, y_imp, _ = train_test_split(\n                X_test_42,\n                y_test_42,\n                train_size=sample_n,\n                stratify=y_test_42,\n                random_state=42,\n            )\n        importance_rows = []\n        for model_name, estimator in seed_42_models.items():\n            print(f"Permutation importance: {model_name}")\n            result = permutation_importance(\n                estimator,\n                X_imp,\n                y_imp,\n                scoring="average_precision",\n                n_repeats=args.importance_repeats,\n                random_state=42,\n                n_jobs=args.n_jobs,\n            )\n            for feature, mean_value, std_value in zip(\n                feature_columns,\n                result.importances_mean,\n                result.importances_std,\n            ):\n                importance_rows.append(\n                    {\n                        "model": model_name,\n                        "feature": feature,\n                        "importance_mean": mean_value,\n                        "importance_std": std_value,\n                        "scoring": "average_precision",\n                        "held_out_sample_records": sample_n,\n                        "permutation_repeats": args.importance_repeats,\n                        "seed": 42,\n                    }\n                )\n        importance_frame = pd.DataFrame(importance_rows)\n        importance_frame.to_csv(\n            output_dir / "permutation_importance.csv",\n            index=False,\n        )\n\n    write_json(output_dir / "model_configurations.json", config_record)\n\n    environment = {\n        "python": sys.version,\n        "platform": platform.platform(),\n        "processor": platform.processor(),\n        "cpu_count": os.cpu_count(),\n        "packages": {\n            "numpy": np.__version__,\n            "pandas": pd.__version__,\n            "scipy": scipy.__version__,\n            "scikit_learn": sklearn.__version__,\n            "xgboost": xgboost.__version__,\n            "joblib": joblib.__version__,\n        },\n    }\n    write_json(output_dir / "environment.json", environment)\n\n    create_plots(\n        test_frame,\n        test_summary,\n        confusion_totals,\n        importance_frame,\n        output_dir,\n    )\n\n    if args.save_models and seed_42_models:\n        model_dir = output_dir / "models_seed_42"\n        model_dir.mkdir(parents=True, exist_ok=True)\n        for model_name, estimator in seed_42_models.items():\n            safe_name = model_name.lower().replace(" ", "_")\n            joblib.dump(estimator, model_dir / f"{safe_name}.joblib")\n\n    manifest = {\n        "status": "completed",\n        "paper_run": (not args.smoke_test and not args.skip_cv\n                      and not args.skip_importance and args.seeds == SEEDS\n                      and args.importance_sample == 20_000\n                      and args.importance_repeats == 3),\n        "dataset_validation": audit["validation_status"],\n        "dataset_sha256": audit["sha256"],\n        "seeds": active_seeds,\n        "cross_validation_performed": not args.skip_cv,\n        "permutation_importance_performed": importance_frame is not None,\n        "elapsed_wall_seconds": time.time() - start_wall,\n        "output_files": sorted(\n            str(path.relative_to(args.workdir))\n            for path in output_dir.rglob("*")\n            if path.is_file()\n        ),\n    }\n    write_json(output_dir / "experiment_manifest.json", manifest)\n\n    print(f"\\nCompleted. Outputs: {output_dir.resolve()}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding='utf-8')
print('Reviewed runner written to:', RUNNER_PATH)


## 5. Run the complete paper experiment

This is a complete-data run using a reduced but defensible computational protocol. No records are discarded. Results are written under `cira_doh_full_run/outputs`.

In [ ]:
import subprocess
import sys
from pathlib import Path

WORKDIR = Path("/content/cira_doh_full_run") if Path("/content").exists() else Path("cira_doh_full_run")

command = [
    sys.executable,
    str(RUNNER_PATH),
    "--data", str(DATA_PATH),
    "--workdir", str(WORKDIR),
    "--n-jobs", "-1",
    "--seeds", "42", "52", "62",
    "--importance-sample", "20000",
    "--importance-repeats", "3",
]

print("Running:", " ".join(command))
subprocess.run(command, check=True)
print("Full experiment completed.")

## 6. Review the principal measured outputs

In [ ]:
import pandas as pd
from IPython.display import display

outputs = WORKDIR / "outputs"

print("Test metrics by seed")
display(pd.read_csv(outputs / "test_metrics_by_seed.csv"))

print("\nSummary across the three seeds")
display(pd.read_csv(outputs / "test_metrics_summary.csv"))

print("\nSummed confusion matrices")
display(pd.read_csv(outputs / "summed_confusion_matrices.csv"))

print("\nLeading permutation-importance features")
importance = pd.read_csv(outputs / "permutation_importance.csv")
display(
    importance.sort_values(
        ["model", "importance_mean"],
        ascending=[True, False],
    ).groupby("model").head(10)
)

## 7. Create a compact results archive

Only the outputs are archived. The 107 MB parquet dataset is excluded.

In [ ]:
import shutil
from pathlib import Path

archive_base = Path("/content/CIRA_DoH_Experiment_Results") if Path("/content").exists() else Path("CIRA_DoH_Experiment_Results")
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=WORKDIR,
    base_dir="outputs",
)
print("Results archive:", archive_path)
print("Archive size:", f"{Path(archive_path).stat().st_size / (1024**2):,.2f} MiB")

## 8. Save or download the results

In Google Colab, run the following cell to download the compact results ZIP. You can then upload that ZIP into the ChatGPT conversation for interpretation and journal writing.

The dataset itself is not downloaded.

In [ ]:
try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print("Not running in Google Colab.")
    print("Use the notebook file browser to download:", archive_path)

### Optional Google Drive backup

Run this only if you prefer to save the results directly to Google Drive.

In [ ]:
# Optional:
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy2(archive_path, '/content/drive/MyDrive/CIRA_DoH_Experiment_Results.zip')
# print('Saved to Google Drive.')